# C7-cnn-transfer — Practice p17 — Solution

Pretrained early and middle features encode edges, textures, and motifs that
generalize across image tasks. ResNet-50's 25.5 million learned parameters
distill experience from roughly a million ImageNet images, far more evidence
than 900 labeled photos could provide from scratch.


In [ ]:
# Cache pin (course convention, plan 009): pretrained weights live in the repo's
# gitignored reference/cache/ -- resolve it from the repo root BEFORE importing torch.
import os, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
             if (p / "pyproject.toml").exists())
os.environ["TORCH_HOME"] = str(_root / "reference" / "cache" / "torch")

import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights

# float32 register (course exception): pretrained resnet50 is a float32 artifact.
# No float64 default here; inputs are cast .to(torch.float32) at the model
# boundary; repeat float32 forwards are bit-identical.
SEED = 20260804

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
model.eval()
assert next(model.parameters()).dtype == torch.float32

import copy
torch.manual_seed(SEED)
x = torch.randn(2, 3, 224, 224).to(torch.float32)

surgical = copy.deepcopy(model)
for p in surgical.parameters():
    p.requires_grad = False
surgical.fc = nn.Linear(2048, 14)
surgical.eval()
with torch.inference_mode():
    out_shape = tuple(surgical(x).shape)
# 2049 * 14, by hand.
hand_head = 28_686
n_trainable = sum(p.numel() for p in surgical.parameters() if p.requires_grad)
head_matches = hand_head == n_trainable


With only 900 examples, a 14-class head with 28,686 scalars is a much more
plausible adjustable surface than the entire 25.5-million-parameter network.
Freezing records that the inherited representation is fixed and only the fresh
head is intended to change; this notebook audits that construction without
performing any fitting procedure.


In [ ]:
# 2049 * 40, by hand.
hand_40 = 81_960


### Answer check

In [ ]:
assert out_shape == (2, 14)
assert hand_head == n_trainable == 28_686
assert head_matches
assert hand_40 == 81_960
assert sorted(name for name, p in surgical.named_parameters() if p.requires_grad) == ["fc.bias", "fc.weight"]
